## Problem (Key Skew): 
Spark partitions data based on the join key.
<br>If one key occurs much more than others (e.g., 90% of rows have the same key), then one partition becomes huge → task skew → slow joins.
<br>This is called key skew.

## Possible Solution:
Append a random number to the skewed key to spread hot keys across multiple partitions.
<br>    * Add n (number of partitions) random numbers (salt) to hot keys in a large dataset. 
<br>    * In a small dataset replicated for each salt

1. Instead of all skewed key rows in one partition, they’re spread across num_salts partitions.
2. Spark can now parallelise join tasks → avoids stragglers.
3. Effective for highly skewed datasets.

## Other solution
1. AQE (Adaptive Query Execution) → Spark can do automatic skew join optimization
2. Aggregate before join → reduce skewed rows

In [31]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .getOrCreate()


In [40]:
# Customers (small dataset)
customers = [(1, "Surya1"), (2, "surya2"), (3, "surya3")]
df_customers = spark.createDataFrame(customers, ["id", "name"])

# Orders (large dataset skewed on customer_id = 1)
orders = [(1, f"order{i}") for i in range(20)] + [(2, "order21"), (3, "order22")]
df_orders = spark.createDataFrame(orders, ["customer_id", "order_name"])


In [41]:
from pyspark.sql.functions import when, floor, rand, lit, col

num_salts = 4
hot_key = 1

df_orders_salted = df_orders.withColumn(
    "salt",
    when(
        col("customer_id") == hot_key,
        floor(rand(seed=42) * num_salts).cast("int")
    ).otherwise(lit(0))
)


In [42]:
df_orders_salted.orderBy("customer_id").show(50)


+-----------+----------+----+
|customer_id|order_name|salt|
+-----------+----------+----+
|          1|   order19|   3|
|          1|    order1|   3|
|          1|    order6|   2|
|          1|    order2|   3|
|          1|    order7|   2|
|          1|    order0|   2|
|          1|    order3|   3|
|          1|   order17|   3|
|          1|    order8|   1|
|          1|   order18|   3|
|          1|   order12|   0|
|          1|   order15|   3|
|          1|    order9|   1|
|          1|    order5|   0|
|          1|   order10|   0|
|          1|   order11|   2|
|          1|    order4|   3|
|          1|   order16|   0|
|          1|   order13|   2|
|          1|   order14|   1|
|          2|   order21|   0|
|          3|   order22|   0|
+-----------+----------+----+



In [43]:
from pyspark.sql.functions import lit
from pyspark.sql import Row

num_salts = 4
hot_key = 1

# Hot customer replicated for each salt
hot_customer = df_customers.filter(col("id") == hot_key)
hot_customer_replicated = spark.createDataFrame(
    [Row(**row.asDict(), salt=i) for row in hot_customer.collect() for i in range(num_salts)]
)

# Non-hot customers salt=0
non_hot_customer = df_customers.filter(col("id") != hot_key).withColumn("salt", lit(0))

# Combine
df_customers_salted = hot_customer_replicated.union(non_hot_customer)


In [45]:
df_customers_salted.show()

+---+------+----+
| id|  name|salt|
+---+------+----+
|  1|Surya1|   0|
|  1|Surya1|   1|
|  1|Surya1|   2|
|  1|Surya1|   3|
|  2|surya2|   0|
|  3|surya3|   0|
+---+------+----+



In [44]:
df_joined = df_customers_salted.join(
    df_orders_salted,
    (df_customers_salted.id == df_orders_salted.customer_id) &
    (df_customers_salted.salt == df_orders_salted.salt),
    how="inner"
)

df_joined.show(100)


+---+------+----+-----------+----------+----+
| id|  name|salt|customer_id|order_name|salt|
+---+------+----+-----------+----------+----+
|  1|Surya1|   0|          1|    order5|   0|
|  1|Surya1|   0|          1|   order10|   0|
|  1|Surya1|   0|          1|   order12|   0|
|  1|Surya1|   0|          1|   order16|   0|
|  1|Surya1|   1|          1|    order8|   1|
|  1|Surya1|   1|          1|    order9|   1|
|  1|Surya1|   1|          1|   order14|   1|
|  1|Surya1|   2|          1|    order0|   2|
|  1|Surya1|   2|          1|    order6|   2|
|  1|Surya1|   2|          1|    order7|   2|
|  1|Surya1|   2|          1|   order11|   2|
|  1|Surya1|   2|          1|   order13|   2|
|  1|Surya1|   3|          1|    order1|   3|
|  1|Surya1|   3|          1|    order2|   3|
|  1|Surya1|   3|          1|    order3|   3|
|  1|Surya1|   3|          1|    order4|   3|
|  1|Surya1|   3|          1|   order15|   3|
|  1|Surya1|   3|          1|   order17|   3|
|  1|Surya1|   3|          1|   or

# Adaptive Query Execution (AQE) with Skew Join